# Squelette générique d'agent IA

Objectif : fournir une base réutilisable pour construire des agents IA.

Structure :
- Configuration
- Mémoire
- Outils
- LLM (mock ou réel)
- Raisonnement
- Exécution
- Boucle principale

Tu pourras remplacer progressivement les éléments mock par OpenAI, LangChain, MCP, RAG ou multi-agents.


In [ ]:
# ===============================
# CONFIGURATION
# ===============================

# MODE_MOCK=True : aucun appel API
# MODE_MOCK=False : utiliser un vrai LLM

MODE_MOCK = True

MODEL_NAME = "gpt-4.1-mini"



In [ ]:
# ===============================
# MÉMOIRE
# ===============================

class Memory:
    '''
    Stockage simple de l'historique.
    
    Plus tard :
    - remplacer par Redis
    - ajouter une mémoire vectorielle
    - ajouter un RAG
    '''
    
    def __init__(self):
        self.history=[]

    def add(self, role, content):
        self.history.append({
            "role": role,
            "content": content
        })

    def get_context(self):
        return self.history

memory=Memory()


In [ ]:
# ===============================
# OUTILS
# ===============================

'''
Les outils représentent les actions disponibles.

Exemples futurs :
- recherche web
- météo
- calendrier
- base SQL
- API métier
'''

def calculator(a,b):
    return a+b


def weather(city):
    return f"Météo simulée : beau temps à {city}"


TOOLS={
    "calculator": calculator,
    "weather": weather
}


In [ ]:
# ===============================
# LLM
# ===============================

class MockLLM:

    '''
    Simule un LLM.
    Permet d'apprendre sans coût API.
    '''

    def generate(self,prompt):

        if "météo" in prompt.lower():
            return {
                "action":"weather",
                "parameters":{
                    "city":"Paris"
                }
            }

        if "addition" in prompt.lower():
            return {
                "action":"calculator",
                "parameters":{
                    "a":5,
                    "b":7
                }
            }

        return {
            "action":None,
            "response":"Réponse textuelle simple"
        }


llm=MockLLM()


In [ ]:
# ===============================
# RAISONNEMENT
# ===============================

class AgentReasoner:

    '''
    Décide :
    
    - faut-il utiliser un outil ?
    - lequel ?
    - avec quels paramètres ?
    '''
    
    def think(self,user_input):

        response=llm.generate(user_input)

        return response


reasoner=AgentReasoner()


In [ ]:
# ===============================
# EXÉCUTION
# ===============================

class AgentExecutor:

    '''
    Exécute les actions demandées.
    '''
    
    def execute(self,decision):

        action=decision.get("action")

        if action is None:
            return decision["response"]

        parameters=decision["parameters"]

        result=TOOLS[action](**parameters)

        return result


executor=AgentExecutor()


In [ ]:
# ===============================
# AGENT PRINCIPAL
# ===============================

class SimpleAgent:

    def __init__(self,memory,reasoner,executor):

        self.memory=memory
        self.reasoner=reasoner
        self.executor=executor


    def run(self,user_input):

        self.memory.add(
            "user",
            user_input
        )

        decision=self.reasoner.think(
            user_input
        )

        result=self.executor.execute(
            decision
        )

        self.memory.add(
            "assistant",
            str(result)
        )

        return result


agent=SimpleAgent(
    memory,
    reasoner,
    executor
)


In [ ]:
# ===============================
# TESTS
# ===============================

print(agent.run(
    "Quelle est la météo ?"
))

print()

print(agent.run(
    "Fais une addition"
))


# Fonctionnement du squelette d'agent IA

Ce notebook implémente un agent IA simple et progressif.

L'objectif est de comprendre comment les différentes parties collaborent avant d'introduire :

- OpenAI API
- LangChain
- MCP
- RAG
- mémoire vectorielle
- multi-agent

---

# Architecture globale

```text
Utilisateur
      │
      ▼

SimpleAgent.run()

      │
      ▼

Memory.add()

      │
      ▼

Reasoner.think()

      │
      ▼

MockLLM.generate()

      │
      ▼

Décision

{
   action,
   parameters
}

      │
      ▼

Executor.execute()

      │
      ▼

Tool

(calculator/weather)

      │
      ▼

Résultat

      │
      ▼

Memory.add()

      │
      ▼

Réponse finale
```

---

# Étape 1 : Configuration

Code :

```python
MODE_MOCK=True
MODEL_NAME="gpt-4.1-mini"
```

Rôle :

Définir les paramètres globaux de l'agent.

Exemple :

- mode développement
- modèle utilisé
- paramètres futurs

---

# Étape 2 : Memory

Code :

```python
memory=Memory()
```

Classe :

```python
class Memory:
```

Rôle :

Stocker l'historique de conversation.

Structure :

```python
[
 {
   "role":"user",
   "content":"Bonjour"
 },
 {
   "role":"assistant",
   "content":"Bonjour"
 }
]
```

L'historique permettra plus tard :

- mémoire longue
- RAG
- stockage Redis
- vector DB

---

# Étape 3 : Outils

Code :

```python
TOOLS={
   "calculator":calculator,
   "weather":weather
}
```

Rôle :

Définir les actions possibles.

Exemple :

```python
calculator(5,7)
```

retourne :

```python
12
```

Exemple :

```python
weather("Paris")
```

retourne :

```python
"Météo simulée : beau temps à Paris"
```

---

# Étape 4 : LLM

Code :

```python
class MockLLM
```

Rôle :

Simuler un vrai LLM sans coût API.

Entrée :

```python
"Quelle météo fait-il ?"
```

Sortie :

```python
{
   "action":"weather",
   "parameters":{
      "city":"Paris"
   }
}
```

Le LLM ne réalise pas l'action.

Il décide seulement :

- répondre directement
- utiliser un outil

---

# Étape 5 : Reasoner

Code :

```python
reasoner.think()
```

Rôle :

Demander au LLM quoi faire.

Flux :

```python
response=llm.generate(
    user_input
)
```

Le résultat est une décision :

```python
{
   "action":"calculator",
   "parameters":{
      "a":5,
      "b":7
   }
}
```

---

# Étape 6 : Executor

Code :

```python
executor.execute()
```

Rôle :

Exécuter les actions.

Flux :

Décision :

```python
{
   "action":"weather"
}
```

↓

Recherche :

```python
TOOLS[action]
```

↓

Exécution :

```python
weather("Paris")
```

↓

Résultat :

```python
"Météo simulée : beau temps à Paris"
```

---

# Étape 7 : Agent principal

Code :

```python
agent.run()
```

Rôle :

Orchestrer tout le système.

Déroulement :

### 1

L'utilisateur envoie :

```python
"Quelle est la météo ?"
```

### 2

Ajout historique :

```python
memory.add(
   "user",
   user_input
)
```

### 3

Le Reasoner réfléchit :

```python
decision=
reasoner.think()
```

### 4

L'Executor agit :

```python
result=
executor.execute()
```

### 5

Ajout résultat :

```python
memory.add(
   "assistant",
   result
)
```

### 6

Retour utilisateur :

```python
print(result)
```

---

# Évolution future

Aujourd'hui :

```text
Utilisateur
→ MockLLM
→ Tool
→ Résultat
```

Semaine suivante :

```text
Utilisateur
→ GPT
→ Tool Calling
→ Résultat
```

Puis :

```text
Utilisateur
→ Memory
→ RAG
→ Agent
→ Tool
→ Résultat
```

Puis :

```text
Utilisateur
→ Planner Agent
→ Research Agent
→ Executor Agent
```

Puis :

```text
Agent A
   │
MCP
   │
Agent B
```

---

Idée importante :

Le notebook actuel est volontairement simple.

Chaque semaine ajoutera une couche supplémentaire sans casser l'architecture existante.

# Exercices

1. Ajouter un outil recherche web
2. Remplacer MockLLM par OpenAI API
3. Ajouter une mémoire vectorielle
4. Ajouter MCP
5. Ajouter plusieurs agents
6. Ajouter FastAPI
7. Ajouter logs et métriques

Ce notebook doit devenir ta base réutilisable pour les futurs exercices.
